# 04. 질의응답 5건 + Before/After 비교 (제출용)

01~03 노트북에서 만든 결과(`extracted.pkl`, `summaries.pkl`)를 불러와
**멀티모달 RAG**와 **텍스트 전용 RAG**에 같은 질문 5개를 던지고 결과를 비교합니다.

- 질문은 모두 **표나 그래프를 봐야 답할 수 있는 것**으로 골랐습니다.
- 파이프라인 코드는 `src/` 모듈(`extract.py`, `summarize.py`, `retriever.py`)에 정리했습니다.

In [1]:
import os, sys, pickle, time
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY 가 없습니다. .env 파일을 확인하세요.")

processed = PROJECT_ROOT / "data" / "processed"
with open(processed / "extracted.pkl", "rb") as f:
    ext = pickle.load(f)
with open(processed / "summaries.pkl", "rb") as f:
    summ = pickle.load(f)

text_chunks, tables = ext["texts_2k_token"], ext["tables"]
text_summaries, table_summaries = summ["text_summaries"], summ["table_summaries"]
image_summaries, img_base64_list = summ["image_summaries"], summ["img_base64_list"]

print(f"텍스트 원본 {len(text_chunks)} / 요약 {len(text_summaries)}")
print(f"표     원본 {len(tables)} / 요약 {len(table_summaries)}")
print(f"이미지 원본 {len(img_base64_list)} / 요약 {len(image_summaries)}")

텍스트 원본 27 / 요약 27
표     원본 35 / 요약 35
이미지 원본 6 / 요약 6


## 1. 검색기와 두 체인 만들기

In [2]:
from src.retriever import (
    build_multivector_retriever, add_documents, build_multimodal_chain,
    build_text_only_chain, is_image,
)

retriever = build_multivector_retriever()
add_documents(retriever, text_summaries, text_chunks)
add_documents(retriever, table_summaries, tables)
add_documents(retriever, image_summaries, img_base64_list)
print("벡터 DB 요약 수:", len(retriever.vectorstore.get()["ids"]))

multimodal_chain = build_multimodal_chain(retriever)   # AFTER
text_only_chain = build_text_only_chain(text_chunks)   # BEFORE
print("체인 준비 완료")

벡터 DB 요약 수: 68


체인 준비 완료


## 2. 질문 5개

각 질문이 원래 어떤 원본(표/이미지)에 근거하는지 함께 적어 둡니다.

In [3]:
queries = [
    ("과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요?", "이미지(표 그림)"),
    ("KOSPI200 지수와 수익률 그래프에서 2008년 전후에 어떤 변화가 나타나나요?", "이미지(그래프)"),
    ("2차 자동조기상환 평가일과 상환금액(액면금액 대비 %)은 얼마인가요?", "표"),
    ("발행사의 2024년 ELS 발행 총계와 발행 잔액은 얼마인가요?", "표"),
    ("기초자산인 KOSPI200 지수의 가격 변동성은 몇 %로 제시되어 있나요?", "표"),
]

table_set = set(tables)
def source_types(docs):
    kinds = []
    for d in docs:
        c = d.page_content if hasattr(d, "page_content") else d
        kinds.append("이미지" if is_image(c) else "표" if c in table_set else "텍스트")
    return kinds

## 3. Before / After 실행

In [4]:
results = []
for i, (q, expected) in enumerate(queries, 1):
    kinds = source_types(retriever.invoke(q))
    before = text_only_chain.invoke(q)
    after = multimodal_chain.invoke(q)
    results.append({"q": q, "expected": expected, "kinds": kinds, "before": before, "after": after})

    print("=" * 80)
    print(f"Q{i}. {q}")
    print(f"   (정답 근거: {expected} / 멀티모달 검색 결과 원본 종류: {', '.join(kinds)})")
    print("-" * 80)
    print("[BEFORE] 텍스트 전용 RAG")
    print(before)
    print("-" * 80)
    print("[AFTER] 멀티모달 RAG")
    print(after)
    print()

Q1. 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요?
   (정답 근거: 이미지(표 그림) / 멀티모달 검색 결과 원본 종류: 이미지, 텍스트, 표, 표)
--------------------------------------------------------------------------------
[BEFORE] 텍스트 전용 RAG
자료에는 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율에 대한 구체적인 내용이 포함되어 있지 않습니다. 또한, 1차 상환 비율에 대한 정보도 제공되지 않았습니다. 따라서 이와 관련된 질문에 대한 답변은 드릴 수 없습니다.
--------------------------------------------------------------------------------
[AFTER] 멀티모달 RAG
상환 차수별 횟수와 빈도율은 다음과 같습니다:

- 1차: 5009회 (97.21%)
- 2차: 80회 (1.55%)
- 3차: 64회 (1.24%)
- 4차: 0회 (0.00%)
- 5차: 0회 (0.00%)

1차 상환 비율은 97.21%입니다.



Q2. KOSPI200 지수와 수익률 그래프에서 2008년 전후에 어떤 변화가 나타나나요?
   (정답 근거: 이미지(그래프) / 멀티모달 검색 결과 원본 종류: 이미지, 이미지, 표, 표)
--------------------------------------------------------------------------------
[BEFORE] 텍스트 전용 RAG
자료에 따르면 KOSPI200 지수와 수익률 그래프에서 2008년 전후에 어떤 변화가 나타나는지에 대한 구체적인 내용은 포함되어 있지 않습니다. 따라서 이 질문에 대한 답변은 모른다고 말씀드릴 수 있습니다.
--------------------------------------------------------------------------------
[AFTER] 멀티모달 RAG
2008년 전후의 KOSPI200 지수와 수익률 그래프에서 나타나는 변화는 다음과 같습니다.

1. **수익률 급등**: 2008년에는 수익률이 급격히 상승하는 모습을 보입니다. 특히 2008년 10월에는 수익률이 9%를 초과하는 지점이 나타났습니다.

2. **KOSPI200 지수 하락**: KOSPI200 지수는 2008년 금융위기와 관련하여 큰 하락세를 보였습니다. 2008년 10월 24일의 종가는 123.27로, 이전에 비해 상당히 낮은 수치입니다.

3. **변동성 증가**: 2008년 전후로 수익률의 변동성이 크게 증가한 것으로 보이며, 이는 금융위기의 영향을 반영하고 있습니다.

이러한 변화는 2008년 글로벌 금융위기와 관련이 깊습니다.



Q3. 2차 자동조기상환 평가일과 상환금액(액면금액 대비 %)은 얼마인가요?
   (정답 근거: 표 / 멀티모달 검색 결과 원본 종류: 표, 표, 텍스트, 표)
--------------------------------------------------------------------------------
[BEFORE] 텍스트 전용 RAG
자료에 따르면, 2차 자동조기상환 평가일과 상환금액(액면금액 대비 %)에 대한 구체적인 내용은 제공되지 않았습니다. 따라서 이 질문에 대한 답변은 모른다고 말씀드릴 수 있습니다.
--------------------------------------------------------------------------------
[AFTER] 멀티모달 RAG
2차 자동조기상환 평가일은 2027년 11월 12일이며, 상환금액은 액면금액 대비 106.10%입니다.



Q4. 발행사의 2024년 ELS 발행 총계와 발행 잔액은 얼마인가요?
   (정답 근거: 표 / 멀티모달 검색 결과 원본 종류: 표, 텍스트, 표, 표)
--------------------------------------------------------------------------------
[BEFORE] 텍스트 전용 RAG
자료에 따르면, 발행사의 2024년 ELS 발행 총계와 발행 잔액에 대한 정보는 제공되지 않았습니다. 따라서 이 질문에 대한 답변은 모른다고 말씀드릴 수 있습니다.
--------------------------------------------------------------------------------
[AFTER] 멀티모달 RAG
자료에 따르면, 2024년 ELS 발행 총계는 23,359억 원이며, 발행 잔액은 2,353억 원입니다.



Q5. 기초자산인 KOSPI200 지수의 가격 변동성은 몇 %로 제시되어 있나요?
   (정답 근거: 표 / 멀티모달 검색 결과 원본 종류: 표, 텍스트, 이미지, 이미지)
--------------------------------------------------------------------------------
[BEFORE] 텍스트 전용 RAG
자료에 기초자산인 KOSPI200 지수의 가격 변동성에 대한 정보는 포함되어 있지 않습니다. 따라서 가격 변동성이 몇 %인지에 대해서는 알 수 없습니다.
--------------------------------------------------------------------------------
[AFTER] 멀티모달 RAG
기초자산인 KOSPI200 지수의 가격 변동성은 34.58%로 제시되어 있습니다.



## 4. README용 질의응답 표

In [5]:
from collections import Counter
from IPython.display import Markdown, display

def one_line(s, n=160):
    s = " ".join(s.split()).replace("|", "/")
    return s if len(s) <= n else s[:n] + "…"

rows = ["| # | 질문 | 멀티모달 RAG 답변 | 근거가 된 원본 종류 |", "| --- | --- | --- | --- |"]
for i, r in enumerate(results, 1):
    c = Counter(r["kinds"])
    kinds = ", ".join(f"{k} {v}" for k, v in c.items())
    rows.append(f"| {i} | {r['q']} | {one_line(r['after'])} | {r['expected']} (검색: {kinds}) |")
display(Markdown("\n".join(rows)))

| # | 질문 | 멀티모달 RAG 답변 | 근거가 된 원본 종류 |
| --- | --- | --- | --- |
| 1 | 과거 데이터 모의실험에서 상환 차수별 횟수와 빈도율은 어떻게 되나요? 1차 상환 비율은 몇 %인가요? | 상환 차수별 횟수와 빈도율은 다음과 같습니다: - 1차: 5009회 (97.21%) - 2차: 80회 (1.55%) - 3차: 64회 (1.24%) - 4차: 0회 (0.00%) - 5차: 0회 (0.00%) 1차 상환 비율은 97.21%입니다. | 이미지(표 그림) (검색: 이미지 1, 텍스트 1, 표 2) |
| 2 | KOSPI200 지수와 수익률 그래프에서 2008년 전후에 어떤 변화가 나타나나요? | 2008년 전후의 KOSPI200 지수와 수익률 그래프에서 나타나는 변화는 다음과 같습니다. 1. **수익률 급등**: 2008년에는 수익률이 급격히 상승하는 모습을 보입니다. 특히 2008년 10월에는 수익률이 9%를 초과하는 지점이 나타났습니다. 2. **KOSPI200 지수 하락*… | 이미지(그래프) (검색: 이미지 2, 표 2) |
| 3 | 2차 자동조기상환 평가일과 상환금액(액면금액 대비 %)은 얼마인가요? | 2차 자동조기상환 평가일은 2027년 11월 12일이며, 상환금액은 액면금액 대비 106.10%입니다. | 표 (검색: 표 3, 텍스트 1) |
| 4 | 발행사의 2024년 ELS 발행 총계와 발행 잔액은 얼마인가요? | 자료에 따르면, 2024년 ELS 발행 총계는 23,359억 원이며, 발행 잔액은 2,353억 원입니다. | 표 (검색: 표 3, 텍스트 1) |
| 5 | 기초자산인 KOSPI200 지수의 가격 변동성은 몇 %로 제시되어 있나요? | 기초자산인 KOSPI200 지수의 가격 변동성은 34.58%로 제시되어 있습니다. | 표 (검색: 표 1, 텍스트 1, 이미지 2) |